## Overview
As an automotive supplier, we are interested in understanding the broader trends in the automotive
industry. For this task, you will create a data pipeline to acquire, process, and load data from various
public sources.

## Objective & Tasks
Your objective is to create a script or a series of scripts (preferably in Python or SQL) that:
**1. Data Acquisition:** Write scripts to download the datasets from the provided public data sources

**2. Data Processing:** Clean and integrate these datasets. This should include, but not be limited to,
handling missing values, duplicates, and possible outliers.

**3. Data Transformation:** Transform the data into a format suitable for further analysis. Justify the
choices you make during this process.

**4. Data Loading:** Write a script to load the data into a hypothetical data storage system. While you
cannot actually load the data into Azure SQL Database or Databricks Delta Lake, you should
simulate the process and include the relevant commands in your script.

**5. Automation Suggestion:** Describe how you would automate this pipeline with a schedule interval
you would choose and explain why.

# Use the following data sources for this task:

U.S. Department of Transportation - National Highway Traffic Safety Administration: Vehicle
Complaints https://www.nhtsa.gov/nhtsa-datasets-and-apis.


# Datacard
**U.S. Department of Transportation - National Highway Traffic Safety Administration: Vehicle
Complaints https://www.nhtsa.gov/nhtsa-datasets-and-apis**

Complaint information entered into NHTSA’s Office of Defects Investigation vehicle owner's complaint database is used with other data sources to identify safety issues that warrant investigation and to determine if a safety-related defect trend exists. Complaint information is also analyzed to monitor existing recalls for proper scope and adequacy.

Schema details: https://static.nhtsa.gov/odi/ffdd/cmpl/CMPL.txt 

##Step 1: Download ZIP file

Use the requests library to download the NHTSA complaints ZIP file from the given URL.

Stream the content in chunks (8192 bytes) to avoid memory issues.

Save the file to DBFS at /dbfs/tmp/FLAT_CMPL.zip.

Print confirmation: "OK - downloadable".

In [0]:
import requests

zip_url = "https://static.nhtsa.gov/odi/ffdd/cmpl/FLAT_CMPL.zip"
local_path = "/dbfs/tmp/FLAT_CMPL.zip"

r = requests.get(zip_url, stream=True)
with open(local_path, "wb") as f:
    for chunk in r.iter_content(8192):
        f.write(chunk)

print("OK - downloadable")

## Step 2: Extract ZIP file

Use Python’s zipfile module to open the downloaded ZIP file.

Extract all contents into the specified folder: /dbfs/tmp/flat_cmpl.

Print confirmation: "OK - extracted".

In [0]:
import zipfile

zip_path = "/dbfs/tmp/FLAT_CMPL.zip"
extract_path = "/dbfs/tmp/flat_cmpl"

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_path)

print("OK - extracted")

## Step 3: Read the data from the decompressed file and checking it's schema

In [0]:
df = (
    spark.read
    .option("header", False)
    .option("inferSchema", True)
    .option("delimiter", "\t")
    .csv("dbfs:/tmp/flat_cmpl/FLAT_CMPL.txt")
)

df.printSchema()



In [0]:
#Creating the bronze DB
#spark.sql("CREATE DATABASE IF NOT EXISTS nhtsa_complaints")


Step 4: Import Libraries and Define Schema 
per doc. https://static.nhtsa.gov/odi/ffdd/cmpl/CMPL.txt

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema = StructType([
    StructField("CMPLID", StringType(), True),           # CHAR(9)
    StructField("ODINO", StringType(), True),            # CHAR(9)
    StructField("MFR_NAME", StringType(), True),         # CHAR(40)
    StructField("MAKETXT", StringType(), True),          # CHAR(25)
    StructField("MODELTXT", StringType(), True),         # CHAR(256)
    StructField("YEARTXT", StringType(), True),          # CHAR(4)
    StructField("CRASH", StringType(), True),            # CHAR(1)
    StructField("FAILDATE", StringType(), True),         # CHAR(8) (YYYYMMDD)
    StructField("FIRE", StringType(), True),             # CHAR(1)
    StructField("INJURED", IntegerType(), True),         # NUMBER
    StructField("FATALITIES", IntegerType(), True),      # NUMBER
    StructField("COMPDESC", StringType(), True),         # CHAR(128)
    StructField("CITY", StringType(), True),             # CHAR(30)
    StructField("STATE", StringType(), True),            # CHAR(15)
    StructField("VIN", StringType(), True),              # CHAR
    StructField("DATEA", StringType(), True),            # CHAR(8) (YYYYMMDD)
    StructField("LDATE", StringType(), True),            # CHAR(8) (YYYYMMDD)
    StructField("MILES", IntegerType(), True),           # NUMBER(7)
    StructField("OCCURENCES", IntegerType(), True),      # NUMBER
    StructField("CDESCR", StringType(), True),           # CHAR(2048)
    StructField("CMPL_TYPE", StringType(), True),        # CHAR(4)
    StructField("POLICE_RPT_YN", StringType(), True),    # CHAR(1)
    StructField("PURCH_DT", StringType(), True),         # CHAR(8)
    StructField("ORIG_OWNER_YN", StringType(), True),    # CHAR(1)
    StructField("ANTI_BRAKES_YN", StringType(), True),   # CHAR(1)
    StructField("CRUISE_CONT_YN", StringType(), True),   # CHAR(1)
    StructField("NUM_CYLS", IntegerType(), True),        # NUMBER
    StructField("DRIVE_TRAIN", StringType(), True),      # CHAR
    StructField("FUEL_SYS", StringType(), True),         # CHAR
    StructField("FUEL_TYPE", StringType(), True),        # CHAR
    StructField("TRANS_TYPE", StringType(), True),       # CHAR(4)
    StructField("VEH_SPEED", IntegerType(), True),       # NUMBER(3)
    StructField("DOT", StringType(), True),              # CHAR(20)
    StructField("TIRE_SIZE", StringType(), True),        # CHAR(30)
    StructField("LOC_OF_TIRE", StringType(), True),      # CHAR(4)
    StructField("TIRE_FAIL_TYPE", StringType(), True),   # CHAR(4)
    StructField("ORIG_EQUIP_YN", StringType(), True),    # CHAR(1)
    StructField("MANUF_DT", StringType(), True),         # CHAR(8)
    StructField("SEAT_TYPE", StringType(), True),        # CHAR(4)
    StructField("RESTRAINT_TYPE", StringType(), True),   # CHAR(4)
    StructField("DEALER_NAME", StringType(), True),      # CHAR
    StructField("DEALER_TEL", StringType(), True),       # CHAR(20)
    StructField("DEALER_CITY", StringType(), True),      # CHAR(30)
    StructField("DEALER_STATE", StringType(), True),     # CHAR(2)
    StructField("DEALER_ZIP", StringType(), True),       # CHAR(10)
    StructField("PROD_TYPE", StringType(), True),        # CHAR(4)
    StructField("REPAIRED_YN", StringType(), True),      # CHAR(1)
    StructField("MEDICAL_ATTN", StringType(), True),     # CHAR(1)
    StructField("VEHICLES_TOWED_YN", StringType(), True) # CHAR(1)
])

## Step 5: Read the new schema and print it

Add a new column Data_update_timestamp dynamically. 

This column will store the current timestamp each time the pipeline runs (including incremental updates).

It automatically captures the current system timestamp when the pipeline runs.
Works for full load and incremental updates.

In [0]:
from pyspark.sql.functions import current_timestamp
df_schema = (

    spark.read
    .option("header", False)
    .option("delimiter", "\t")
    .schema(schema)
    .csv("dbfs:/tmp/flat_cmpl/FLAT_CMPL.txt")
)


# Add timestamp column
df_with_timestamp = df_schema.withColumn("Data_update_timestamp", current_timestamp())

# Show schema and sample
df_with_timestamp.printSchema()
display(df_with_timestamp)







## Step 6: Write the DataFrame to Bronze Delta Table



In [0]:
# Define Bronze path
bronze_path = "dbfs:/mnt/datalake/bronze/nhtsa_complaints"

# Write DataFrame in Delta format
df_with_timestamp.write.format("delta").mode("overwrite").option("mergeSchema", "true").save(bronze_path)

print(f"Bronze table saved at {bronze_path}")

## Step 7: Register the bronze table in the metastore for SQL access



In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS nhtsa_complaints.bronze
USING DELTA
LOCATION '{bronze_path}'
""")

## Step 8: Display the bronze table

In [0]:
spark.read.table("hive_metastore.nhtsa_complaints.bronze").display()